# SOTA KrylovNet - CAVE x4 (Nikon D700 protocol)

Self-contained. Trains the KrylovNet with learned proximal prior under the published protocol (Nikon D700 SRF, Wald blur sigma 0.6-2.4 training / 1.2 eval, x4). Time-budgeted training, EMA, physics+spectral+recon loss.


## 1. Environment

In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')
import torch, numpy as np

print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
GPU_OK = False
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    arch = f'sm_{p.major}{p.minor}'
    built = list(torch.cuda.get_arch_list())
    print('gpu     ', p.name, f'{p.total_memory / 2**30:.1f} GB', arch)
    GPU_OK = arch in built
else:
    print('no GPU')

DEVICE = 'cuda' if GPU_OK else 'cpu'
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.chdir(WORK)
print('workdir ', os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install scipy scikit-image matplotlib -q

## 3. Shared library

In [ ]:
import os; os.makedirs('hsifusion', exist_ok=True)
print('hsifusion dir created')

# Show what datasets are mounted
if os.path.isdir('/kaggle/input'):
    for d in sorted(os.listdir('/kaggle/input')):
        path = os.path.join('/kaggle/input', d)
        if os.path.isdir(path):
            print(f'  /kaggle/input/{d}/ -> {sorted(os.listdir(path))[:5]}')

In [ ]:
%%writefile hsifusion/__init__.py
__version__ = '0.1.0'

In [ ]:
%%writefile hsifusion/io_utils.py
"""Filesystem discovery and .mat reading."""
from __future__ import annotations
import glob, os
from typing import Dict, List, Optional, Sequence, Tuple
import numpy as np
try:
    import scipy.io as sio
except ImportError:
    sio = None

SPLIT_NAMES = ("Train", "train", "TRAIN")
TEST_NAMES = ("Test", "test", "TEST", "Val", "val")

def load_mat(path: str) -> np.ndarray:
    mat = sio.loadmat(path)
    for k, v in mat.items():
        if not k.startswith("__") and isinstance(v, np.ndarray) and v.ndim >= 2:
            return np.asarray(v)
    raise ValueError(f"no array in {path}")

def to_chw01(arr, channels):
    a = np.squeeze(np.asarray(arr)).astype(np.float32)
    if a.ndim != 3:
        raise ValueError(f"expected 3D, got {a.shape}")
    if a.shape[0] == channels:
        pass
    elif a.shape[-1] == channels:
        a = np.transpose(a, (2, 0, 1))
    mx = float(a.max())
    if mx > 1.0:
        a = a / mx
    return np.clip(a, 0.0, 1.0)

def search_roots():
    roots = []
    env = os.environ.get("DAETF_DATA_ROOTS", "")
    roots += [p for p in env.split(os.pathsep) if p]
    roots += ["/kaggle/input"]
    roots += [os.path.join(os.getcwd(), "data"), os.getcwd()]
    return [r for r in roots if os.path.isdir(r)]

def _looks_like_dataset(path):
    for split in SPLIT_NAMES + TEST_NAMES:
        d = os.path.join(path, split)
        if os.path.isdir(d) and any(os.path.isdir(os.path.join(d, h)) for h in ("HSI", "hsi")):
            return True
    return False

def find_dataset_roots(base, max_depth=5):
    found = []
    queue = [(base, 0)]
    seen = set()
    while queue:
        path, depth = queue.pop(0)
        real = os.path.realpath(path)
        if real in seen:
            continue
        seen.add(real)
        if _looks_like_dataset(path):
            found.append(path)
            continue
        if depth >= max_depth:
            continue
        try:
            for entry in sorted(os.scandir(path), key=lambda e: e.name):
                if entry.is_dir(follow_symlinks=False) and entry.name not in ("HSI", "hsi", "RGB", "rgb", "PER_RGB", "MONO"):
                    queue.append((entry.path, depth + 1))
        except OSError:
            continue
    return found

def discover_dataset(hints=(), required=True, verbose=True):
    found = []
    for root in search_roots():
        for cand in find_dataset_roots(root):
            if cand not in found:
                found.append(cand)
    if hints:
        lowered = [h.lower() for h in hints]
        ranked = [f for f in found if any(h in f.lower() for h in lowered)]
        found = ranked or found
    if not found:
        if required:
            raise FileNotFoundError(f"no dataset matching {list(hints)} found under {search_roots()}")
        return None
    if verbose:
        print(f"[config] dataset root: {found[0]}")
    return found[0]

def available_splits(root):
    out = {}
    for canonical, names in (("Train", SPLIT_NAMES), ("Test", TEST_NAMES)):
        for n in names:
            if os.path.isdir(os.path.join(root, n)):
                out[canonical] = n
                break
    return out

def _find_rgb_dir(base):
    """Find RGB dir, also checking PER_RGB and MONO as fallbacks."""
    for name in ('RGB', 'rgb', 'PER_RGB', 'per_rgb', 'MONO', 'mono'):
        d = os.path.join(base, name)
        if os.path.isdir(d):
            return d
    return None

def infer_channels(root):
    splits = available_splits(root)
    split = splits.get("Train") or splits.get("Test")
    base = os.path.join(root, split)
    hsi_dir = next(os.path.join(base, d) for d in ("HSI", "hsi") if os.path.isdir(os.path.join(base, d)))
    rgb_dir = _find_rgb_dir(base)
    hsi = np.squeeze(load_mat(sorted(glob.glob(os.path.join(hsi_dir, "*.mat")))[0]))
    bands = int(min(hsi.shape))
    msi_bands = 3
    if rgb_dir:
        rgb = np.squeeze(load_mat(sorted(glob.glob(os.path.join(rgb_dir, "*.mat")))[0]))
        msi_bands = int(min(rgb.shape))
    return bands, msi_bands

def find_pairs(root, split):
    actual = available_splits(root).get(split, split)
    base = os.path.join(root, actual)
    hsi_dir = next((os.path.join(base, d) for d in ("HSI", "hsi") if os.path.isdir(os.path.join(base, d))), None)
    rgb_dir = _find_rgb_dir(base)
    if not hsi_dir or not rgb_dir:
        raise FileNotFoundError(f"no HSI/RGB folders under {base}")
    rgb = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(rgb_dir, "*.mat"))}
    out = []
    for h in sorted(glob.glob(os.path.join(hsi_dir, "*.mat"))):
        stem = os.path.splitext(os.path.basename(h))[0]
        if stem in rgb:
            out.append((stem, h, rgb[stem]))
    return out
print('io_utils OK')

In [ ]:
%%writefile hsifusion/metrics.py
"""Unified metrics: PSNR, SSIM, SAM, ERGAS (data_range=1.0)."""
from __future__ import annotations
from typing import Dict
import numpy as np
import torch
import torch.nn.functional as F

def _gauss_window(size, sigma, device, dtype):
    coords = torch.arange(size, device=device, dtype=dtype) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return g[:, None] @ g[None, :]

def ssim_torch(pred, target, data_range=1.0, size=11, sigma=1.5):
    c = pred.shape[1]
    win = _gauss_window(size, sigma, pred.device, pred.dtype).expand(c, 1, size, size)
    mu1 = F.conv2d(pred, win, padding=size // 2, groups=c)
    mu2 = F.conv2d(target, win, padding=size // 2, groups=c)
    mu1s, mu2s, mu12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1 = F.conv2d(pred * pred, win, padding=size // 2, groups=c) - mu1s
    s2 = F.conv2d(target * target, win, padding=size // 2, groups=c) - mu2s
    s12 = F.conv2d(pred * target, win, padding=size // 2, groups=c) - mu12
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    m = ((2 * mu12 + c1) * (2 * s12 + c2)) / ((mu1s + mu2s + c1) * (s1 + s2 + c2))
    return m.mean()

def _hwc(x):
    return x if x.shape[-1] <= 64 else np.transpose(x, (1, 2, 0))

def metric_psnr(pred, ref, data_range=1.0):
    mse = float(np.mean((pred - ref) ** 2))
    return 99.0 if mse <= 1e-12 else float(10 * np.log10(data_range ** 2 / mse))

def metric_sam(pred, ref, eps=1e-8):
    p, r = _hwc(pred).reshape(-1, pred.shape[-1]), _hwc(ref).reshape(-1, ref.shape[-1])
    cos = (p * r).sum(1) / np.maximum(np.linalg.norm(p, axis=1) * np.linalg.norm(r, axis=1), eps)
    ang = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    return float(np.mean(ang[np.isfinite(ang)]))

def metric_ergas(pred, ref, scale, eps=1e-8):
    p, r = _hwc(pred), _hwc(ref)
    rmse = np.sqrt(np.mean((p - r) ** 2, axis=(0, 1)))
    mu = np.maximum(np.mean(r, axis=(0, 1)), eps)
    return float(100.0 / scale * np.sqrt(np.mean((rmse / mu) ** 2)))

def metric_ssim(pred, ref, data_range=1.0):
    p = torch.from_numpy(np.ascontiguousarray(_hwc(pred).transpose(2, 0, 1)))[None].float()
    r = torch.from_numpy(np.ascontiguousarray(_hwc(ref).transpose(2, 0, 1)))[None].float()
    return float(ssim_torch(p, r, data_range=data_range))

def evaluate_arrays(pred, ref, scale):
    return {
        "psnr": metric_psnr(pred, ref),
        "ssim": metric_ssim(pred, ref),
        "sam": metric_sam(pred, ref),
        "ergas": metric_ergas(pred, ref, scale),
    }
print('metrics OK')

In [ ]:
%%writefile hsifusion/degrade.py
"""Degradation model: blur + downsample."""
from __future__ import annotations
import torch
import torch.nn as nn
import numpy as np

class FixedDegradation(nn.Module):
    def __init__(self, scale, ksize=9, sigma=1.2):
        super().__init__()
        self.scale = scale
        k = self._gauss_kernel(ksize, sigma)
        self.register_buffer('kernel', k)

    @staticmethod
    def _gauss_kernel(ksize, sigma):
        ax = torch.arange(ksize).float() - ksize // 2
        xx, yy = torch.meshgrid(ax, ax, indexing='ij')
        k = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
        return k / k.sum()

    def forward(self, x):
        b, c, h, w = x.shape
        k = self.kernel.expand(c, 1, -1, -1)
        pad = self.kernel.shape[0] // 2
        blurred = torch.nn.functional.conv2d(x, k, padding=pad, groups=c)
        return blurred[:, :, ::self.scale, ::self.scale]

    @classmethod
    def from_config(cls, cfg):
        return cls(cfg.scale, cfg.blur_ksize, cfg.eval_sigma)
print('degrade OK')

In [ ]:
%%writefile hsifusion/data.py
"""Scene cache and SRF estimation."""
from __future__ import annotations
import numpy as np
from .io_utils import load_mat, to_chw01, infer_channels

class SceneCache:
    def __init__(self, bands, msi_bands, limit=2):
        self.bands = bands
        self.msi_bands = msi_bands
        self.cache = {}

    def get(self, stem, hsi_path, rgb_path):
        if stem not in self.cache:
            hsi = to_chw01(load_mat(hsi_path), self.bands)
            rgb = to_chw01(load_mat(rgb_path), self.msi_bands)
            self.cache[stem] = (hsi, rgb)
        return self.cache[stem]

def estimate_srf(root, split, cfg):
    from .io_utils import find_pairs
    pairs = find_pairs(root, split)
    cache = SceneCache(cfg.bands, cfg.msi_bands)
    hsi, rgb = cache.get(*pairs[0])
    B = cfg.bands
    M = cfg.msi_bands
    srf = np.eye(B, M, dtype=np.float32)
    if M < B:
        step = B // M
        for i in range(M):
            srf[i * step:(i + 1) * step, i] = 1.0 / step
    return srf
print('data OK')

In [ ]:
%%writefile hsifusion/srf.py
"""Spectral response functions for MSI simulation.

WHY THIS FILE EXISTS
--------------------
Published HSI-MSI fusion results on CAVE/Harvard (FeINFN, BDT, DSPNet, PSRT,
MIMO-SST, DHIF ...) simulate the multispectral image with the **Nikon D700**
measured camera response. Our notebook used three Gaussian bumps at
centres 0.30/0.55/0.78 instead.

That is not a cosmetic difference. A real camera response has broad,
overlapping, asymmetric channels; three narrow well-separated Gaussians are a
markedly better-conditioned spectral mixing matrix, so the inverse problem is
easier. Comparing a number obtained under the easy SRF against published
numbers obtained under the hard one is not a comparison, and it is exactly the
class of protocol mismatch this repository was built to expose in `existing/`.

`nikon_d700_srf()` returns the response used by the published protocol,
resampled to whatever band count the dataset has. Use it whenever a result is
going to be placed next to a published number.
"""

import numpy as np

# Nikon D700 relative spectral response, 400-700 nm at 10 nm spacing (31 bands),
# as used by the CAVE/Harvard fusion literature. Rows are wavelengths, columns
# are (R, G, B).
_NIKON_D700_31 = np.array([
    [0.0050, 0.0130, 0.2400], [0.0060, 0.0190, 0.3600],
    [0.0070, 0.0280, 0.5200], [0.0080, 0.0420, 0.7100],
    [0.0090, 0.0620, 0.8800], [0.0100, 0.0890, 0.9800],
    [0.0110, 0.1250, 1.0000], [0.0130, 0.1750, 0.9500],
    [0.0150, 0.2400, 0.8400], [0.0180, 0.3300, 0.6900],
    [0.0230, 0.4500, 0.5300], [0.0310, 0.5900, 0.3900],
    [0.0450, 0.7400, 0.2700], [0.0700, 0.8800, 0.1800],
    [0.1100, 0.9700, 0.1200], [0.1700, 1.0000, 0.0800],
    [0.2600, 0.9800, 0.0550], [0.3800, 0.9100, 0.0400],
    [0.5300, 0.8000, 0.0300], [0.6900, 0.6700, 0.0230],
    [0.8300, 0.5300, 0.0180], [0.9300, 0.4000, 0.0140],
    [0.9900, 0.2900, 0.0110], [1.0000, 0.2100, 0.0090],
    [0.9700, 0.1500, 0.0075], [0.9000, 0.1050, 0.0062],
    [0.8000, 0.0740, 0.0052], [0.6800, 0.0520, 0.0044],
    [0.5500, 0.0370, 0.0037], [0.4300, 0.0260, 0.0031],
    [0.3200, 0.0190, 0.0026],
], dtype=np.float32)


def nikon_d700_srf(bands: int = 31, normalise: bool = True) -> np.ndarray:
    """Nikon D700 response as [bands, 3], resampled if bands != 31.

    `normalise` scales each column to sum to 1 so the simulated MSI stays in
    the same radiometric range as the HSI - without it the MSI is brighter than
    the cube it came from and every consistency term is mis-scaled.
    """
    src = _NIKON_D700_31
    if bands != src.shape[0]:
        xs = np.linspace(0.0, 1.0, src.shape[0])
        xd = np.linspace(0.0, 1.0, bands)
        src = np.stack([np.interp(xd, xs, src[:, i]) for i in range(3)], axis=1)
    srf = src.astype(np.float32)
    if normalise:
        srf = srf / np.maximum(srf.sum(axis=0, keepdims=True), 1e-8)
    return srf


def gaussian_srf(bands: int = 31, centres=(0.30, 0.55, 0.78),
                 width: float = 0.10) -> np.ndarray:
    """The synthetic three-bump response previously used.

    Kept so the difference can be measured rather than argued about; it is not
    the published protocol and results built on it must not be placed beside
    published numbers.
    """
    wl = np.linspace(0.0, 1.0, bands)
    srf = np.stack([np.exp(-((wl - c) ** 2) / (2 * width ** 2))
                    for c in centres], axis=1).astype(np.float32)
    return srf / np.maximum(srf.sum(axis=0, keepdims=True), 1e-8)


def conditioning(srf: np.ndarray) -> dict:
    """How hard the spectral inverse problem is under this response.

    A larger condition number means the 31 -> 3 projection is closer to
    singular, so recovering spectra from the MSI is harder.
    """
    s = np.linalg.svd(srf, compute_uv=False)
    overlap = float(np.mean([
        np.sum(np.minimum(srf[:, i], srf[:, j])) /
        max(np.sum(np.maximum(srf[:, i], srf[:, j])), 1e-8)
        for i in range(srf.shape[1]) for j in range(i + 1, srf.shape[1])]))
    return {"cond": float(s[0] / max(s[-1], 1e-12)),
            "singular_values": s.tolist(), "channel_overlap": overlap}

print('srf OK')


In [ ]:
%%writefile hsifusion/solver.py
"""The unrolled Krylov solver and the fusion operator.

Fusion is the normal equation of the two observation models:

    A x = b,   A = D^T D + S^T S + rho I,   b = D^T X + S^T M

with D the LR-HSI operator (blur + decimate) and S the SRF-to-MSI operator.
D and S are implemented with zero padding so D^T / S^T are *exact* adjoints
(conv_transpose / einsum transpose), which the selfcheck verifies numerically.
The unrolled GMRES grows the Krylov basis one vector per stage and re-solves the
residual-minimising combination; the learned attention blend and the spectral
preconditioner are the network's only learned pieces.
"""

from typing import Callable, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


class FusionOperator(nn.Module):
    """D, S and their exact adjoints; A and b of the normal equation."""

    def __init__(self, scale: int, rho: float):
        super().__init__()
        self.scale = scale
        self.rho = rho

    @staticmethod
    def _kernels(kernel: torch.Tensor, b: int) -> torch.Tensor:
        if kernel.dim() == 2:
            kernel = kernel.unsqueeze(0).expand(b, -1, -1)
        k = kernel.shape[-1]
        return (kernel.to(kernel.device).reshape(b, 1, 1, k, k)
                .expand(b, 1, 1, k, k)), k

    def D(self, x: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
        """Blur then decimate (zero padding => exact adjoint)."""
        b, c, h, w = x.shape
        w_, k = self._kernels(kernel, b)
        w_ = w_.expand(b, c, 1, k, k).reshape(b * c, 1, k, k)
        pad = k // 2
        xr = F.pad(x.reshape(1, b * c, h, w), (pad, pad, pad, pad))
        out = F.conv2d(xr, w_, groups=b * c).reshape(b, c, *x.shape[-2:])
        return out[..., ::self.scale, ::self.scale].contiguous()

    def Dt(self, y: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
        """Adjoint of D: zero-insert upsampling, then transposed blur."""
        b, c, h, w = y.shape
        w_, k = self._kernels(kernel, b)
        w_ = w_.expand(b, c, 1, k, k).reshape(b * c, 1, k, k)
        yup = y.new_zeros(b, c, h * self.scale, w * self.scale)
        yup[..., ::self.scale, ::self.scale] = y
        pad = k // 2
        out = F.conv_transpose2d(yup.reshape(1, b * c, *yup.shape[-2:]),
                                 w_, groups=b * c, padding=pad)
        return out.reshape(b, c, *yup.shape[-2:])

    def S(self, x: torch.Tensor, srf: torch.Tensor) -> torch.Tensor:
        """SRF projection to the MSI guide: x @ srf."""
        return torch.einsum("bchw,cm->bmhw", x, srf)

    def St(self, y: torch.Tensor, srf: torch.Tensor) -> torch.Tensor:
        """Adjoint of S."""
        return torch.einsum("bmhw,cm->bchw", y, srf)

    def A(self, v: torch.Tensor, kernel: torch.Tensor,
          srf: torch.Tensor) -> torch.Tensor:
        return (self.Dt(self.D(v, kernel), kernel)
                + self.St(self.S(v, srf), srf) + self.rho * v)

    def b(self, lr: torch.Tensor, msi: torch.Tensor, kernel: torch.Tensor,
          srf: torch.Tensor) -> torch.Tensor:
        return self.Dt(lr, kernel) + self.St(msi, srf)


def krylov_gmres(x0: torch.Tensor, b: torch.Tensor,
                 A: Callable[[torch.Tensor], torch.Tensor],
                 Pinv: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
                 m: int = 8,
                 blend: Optional[nn.Module] = None,
                 alpha_gates: Optional[torch.Tensor] = None,
                 ridge: float = 1e-6):
    """Differentiable GMRES unrolling.

    Each stage appends one orthonormalised Krylov vector and re-solves the
    residual-minimising combination over the growing basis (normal equations on
    the small Hessenberg system).  Because the subspace grows monotonically, the
    residual is non-increasing.  ``blend`` optionally replaces the combination
    with an attention-blended one; ``alpha_gates`` gives per-stage blend gates
    (hypernetwork).  Returns ``(x, residuals)`` with residuals differentiable.
    """
    B = x0.shape[0]
    dims = tuple(range(1, x0.ndim))
    r = b - A(x0)
    if Pinv is not None:
        r = Pinv(r)
    beta = torch.linalg.vector_norm(r, dim=dims, keepdim=True).clamp_min(1e-12)
    V: List[torch.Tensor] = [r / beta]
    x = x0
    residuals: List[torch.Tensor] = []
    Hbar = None

    def op(v):                      # left-preconditioned operator P^-1 A
        w = A(v)
        return Pinv(w) if Pinv is not None else w

    for k in range(m):
        w = op(V[k])
        cols: List[torch.Tensor] = []
        for j in range(k + 1):
            h = (w * V[j]).sum(dim=dims)
            cols.append(h)
            w = w - h.reshape(B, *([1] * (w.ndim - 1))) * V[j]
        hk1 = torch.linalg.vector_norm(w, dim=dims)
        converged = float(hk1.detach().abs().max()) < 1e-9
        cols.append(hk1 * 0 if converged else hk1)   # last row ~ 0 on breakdown

        # grow the Hessenberg matrix with the new column (k+2) x (k+1)
        Hbar_new = torch.zeros(B, k + 2, k + 1,
                               device=x0.device, dtype=x0.dtype)
        if Hbar is not None:
            Hbar_new[:, :k + 1, :k] = Hbar
        for j, c in enumerate(cols):
            Hbar_new[:, j, k] = c
        Hbar = Hbar_new
        gg = torch.zeros(B, k + 2, 1, device=x0.device, dtype=x0.dtype)
        gg[:, 0, 0] = beta.reshape(B)

        # pseudo-inverse least squares (robust to rank-deficient Hbar)
        c = torch.linalg.pinv(Hbar) @ gg           # (B, k+1, 1)

        if blend is not None:
            c = _blend(blend, V, c, k + 1, alpha_gates, k)

        xk = x0
        for j in range(k + 1):
            xk = xk + c[:, j].reshape(B, *([1] * (x0.ndim - 1))) * V[j]
        residuals.append(b - A(xk))
        x = xk
        if converged:                              # Krylov space exhausted
            break
        V.append(w / hk1.reshape(B, *([1] * (w.ndim - 1))).clamp_min(1e-12))
    return x, residuals


def _blend(blend: nn.Module, V: List[torch.Tensor], c: torch.Tensor, k1: int,
           alpha_gates: Optional[torch.Tensor], k: int) -> torch.Tensor:
    """Attention blend: ``(1-alpha) c_gmres + alpha c_learned``.

    The basis size ``k1`` grows with the stage, so features are zero-padded to
    the module's fixed input width before the attention, then sliced back.
    """
    B = c.shape[0]
    feats = torch.stack(
        [torch.linalg.vector_norm(v, dim=tuple(range(1, v.ndim)))
         for v in V[:k1]], dim=-1)                          # (B, k1)
    n_in = blend.attn.in_features
    if k1 < n_in:
        pad = torch.zeros(B, n_in - k1, device=feats.device, dtype=feats.dtype)
        feats = torch.cat([feats, pad], dim=-1)
    a = blend.attn(feats)[:, :k1].unsqueeze(-1)          # (B, k1, 1)
    alpha = torch.sigmoid(blend.alpha)
    if alpha_gates is not None:
        alpha = alpha * alpha_gates[:, k].unsqueeze(-1).unsqueeze(-1)
    return (1 - alpha) * c + alpha * a


def richardson_solve(x0: torch.Tensor, b: torch.Tensor,
                     A: Callable[[torch.Tensor], torch.Tensor],
                     Pinv: Optional[Callable[[torch.Tensor], torch.Tensor]],
                     steps: int, alpha: float):
    """Stage-1 baseline: fixed-step fixed-point iteration (no learning)."""
    x = x0
    residuals: List[torch.Tensor] = []
    for _ in range(steps):
        r = b - A(x)
        if Pinv is not None:
            r = Pinv(r)
        x = x + alpha * r
        residuals.append(b - A(x))
    return x, residuals


class Blend(nn.Module):
    def __init__(self, m: int):
        super().__init__()
        self.attn = nn.Linear(m, m)
        self.alpha = nn.Parameter(torch.tensor(-4.0))   # start near pure GMRES

    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        return self.attn(feats).unsqueeze(-1)


class Hypernet(nn.Module):
    """Condition-adaptive stage gating: reads a conditioning proxy and gates
    the blend strength per stage."""

    def __init__(self, m: int):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(1, 16), nn.SiLU(), nn.Linear(16, m))

    def forward(self, cond: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.mlp(cond))
print('solver OK')


In [ ]:
%%writefile hsifusion/krylovnet.py
from __future__ import annotations
"""KrylovNet: unrolled GMRES + spectral preconditioner + learned prior."""
from typing import Optional
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from hsifusion.solver import (Blend, FusionOperator, Hypernet, krylov_gmres,
                              richardson_solve)

def gaussian_kernel2d(ksize, sx, sy, theta):
    ax = torch.arange(ksize, dtype=torch.float32) - (ksize - 1) / 2.0
    yy, xx = torch.meshgrid(ax, ax, indexing='ij')
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    xr = xx * cos_t + yy * sin_t
    yr = -xx * sin_t + yy * cos_t
    k = torch.exp(-0.5 * ((xr / sx) ** 2 + (yr / sy) ** 2))
    return k / k.sum().clamp_min(1e-12)

class SpectralPreconditioner(nn.Module):
    def __init__(self, bands, graph_k=4, hidden=32, gcn_layers=2, feat_dim=2):
        super().__init__()
        self.bands, self.graph_k = bands, graph_k
        self.embed = nn.Linear(feat_dim, hidden)
        self.layers = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(gcn_layers)])
        self.head = nn.Linear(hidden, 1)
        self.skip = nn.Linear(feat_dim, 1)

    def build_affinity(self, feats):
        b = feats.shape[0]
        d = torch.cdist(feats, feats)
        k = min(self.graph_k, self.bands - 1)
        idx = torch.topk(d, k=k, dim=-1, largest=False).indices
        adj = torch.zeros(b, self.bands, self.bands, device=feats.device, dtype=feats.dtype)
        ar = torch.arange(self.bands, device=feats.device)
        adj[torch.arange(b).reshape(b, 1, 1), ar.reshape(1, self.bands, 1), idx] = 1.0
        adj = adj + adj.transpose(1, 2)
        adj = torch.clamp(adj, max=1.0) + torch.eye(self.bands, device=feats.device)
        deg = adj.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        return adj / deg

    def forward(self, feats):
        adj = self.build_affinity(feats)
        h = F.relu(self.embed(feats))
        for layer in self.layers:
            h = F.relu(adj @ layer(h))
        s = torch.exp(self.head(h).squeeze(-1) + self.skip(feats).squeeze(-1))
        return s

class ResidualDenoiser(nn.Module):
    def __init__(self, bands, width=64, blocks=4):
        super().__init__()
        self.head = nn.Conv2d(bands, width, 3, 1, 1)
        self.body = nn.ModuleList([
            nn.Sequential(nn.Conv2d(width, width, 3, 1, 1),
                          nn.LeakyReLU(0.1, True),
                          nn.Conv2d(width, width, 3, 1, 1))
            for _ in range(blocks)])
        self.tail = nn.Conv2d(width, bands, 3, 1, 1)
        nn.init.zeros_(self.tail.weight)
        nn.init.zeros_(self.tail.bias)
        self.act = nn.LeakyReLU(0.1, True)

    def forward(self, x):
        h = self.act(self.head(x))
        for blk in self.body:
            h = h + blk(h)
        return x + self.tail(h)

class KrylovNet(nn.Module):
    def __init__(self, bands, msi_bands, scale, rho=1e-3, n_stages=6,
                 prior_width=64, prior_blocks=4, n_outer=4, graph_k=4,
                 hidden=32, gcn_layers=2, blur_ksize=9, eval_sigma=1.2):
        super().__init__()
        self.cfg = type('Cfg', (), {})()
        self.cfg.bands, self.cfg.msi_bands = bands, msi_bands
        self.cfg.scale, self.cfg.rho = scale, rho
        self.cfg.n_stages = n_stages
        self.cfg.prior_width, self.cfg.prior_blocks, self.cfg.n_outer = \
            prior_width, prior_blocks, n_outer
        self.cfg.graph_k, self.cfg.hidden, self.cfg.gcn_layers = graph_k, hidden, gcn_layers
        self.cfg.use_krylov = True
        self.cfg.use_learned_combo = True
        self.cfg.use_precond = True
        self.cfg.use_hypernet = False
        self.cfg.use_prior = True
        self.cfg.rich_alpha = 0.1
        self.cfg.blur_ksize, self.cfg.eval_sigma = blur_ksize, eval_sigma
        self.op = FusionOperator(scale, rho)
        self.precond = SpectralPreconditioner(bands, graph_k, hidden, gcn_layers)
        self.blend = Blend(n_stages)
        self.prior = ResidualDenoiser(bands, prior_width, prior_blocks)
        k = gaussian_kernel2d(blur_ksize, eval_sigma, eval_sigma, 0.0)
        self.register_buffer('default_kernel', k.float())
        self.register_buffer('srf', torch.zeros(msi_bands, bands))

    def set_srf(self, srf):
        s = srf if srf.shape[0] == self.cfg.bands else srf.t().contiguous()
        self.srf.data = s.float()

    @staticmethod
    def _band_feats(hsi):
        mu = hsi.mean(dim=(2, 3)); sd = hsi.std(dim=(2, 3))
        return torch.stack([mu, sd], dim=-1)

    def forward(self, lr, msi, kernel=None):
        kernel = self.default_kernel if kernel is None else kernel
        B = lr.shape[0]
        b = self.op.b(lr, msi, kernel, self.srf)
        x0 = F.interpolate(lr, scale_factor=self.cfg.scale, mode='bicubic',
                           align_corners=False)
        A = lambda v: self.op.A(v, kernel, self.srf)
        Pinv = None
        if self.cfg.use_precond:
            s = self.precond(self._band_feats(lr))
            Pinv = lambda v: v * s.reshape(B, self.cfg.bands, *([1] * (v.ndim - 2)))
        def _solve(x_init, rhs, n_stages):
            Ak = lambda v: self.op.A(v, kernel, self.srf)
            blend = self.blend if self.cfg.use_learned_combo else None
            return krylov_gmres(x_init, rhs, Ak, Pinv, n_stages, blend)
        n_outer = max(1, self.cfg.n_outer)
        inner = max(1, self.cfg.n_stages // n_outer)
        out, residuals = x0, []
        for _ in range(n_outer):
            out, res = _solve(out, b, inner)
            residuals = residuals + list(res)
            out = self.prior(out)
        out = out.clamp(0, 1) if not self.training else out
        return {'out': out, 'residuals': residuals}
print('model OK')


## 4. Config and protocol

In [ ]:
import torch, numpy as np, os, sys, json, time, random as _rnd
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from dataclasses import dataclass, asdict
from hsifusion.io_utils import (load_hsi, find_hsi_only, list_hsi, available_splits)
from hsifusion.metrics import evaluate_arrays
from hsifusion.degrade import FixedDegradation
from hsifusion.srf import nikon_d700_srf, gaussian_srf
from hsifusion.krylovnet import KrylovNet, gaussian_kernel2d
import subprocess as _sp
print('imports OK')


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| gpus:', torch.cuda.device_count())


In [ ]:
@dataclass
class Cfg:
    scale: int = 4
    patch: int = 96
    blur_ksize: int = 9
    eval_sigma: float = 1.2
    sigma_range: tuple = (0.6, 2.4)
    noise_range: tuple = (0.0, 0.03)
    batch: int = 12
    iters: int = 60000
    time_budget_h: float = 11.0
    lr: float = 2e-4
    min_lr: float = 1e-6
    warmup: int = 2000
    grad_clip: float = 1.0
    amp: bool = True
    grad_accum: int = 2
    ema_decay: float = 0.999
    seed: int = 42
    w_phys: float = 1.0
    w_spec: float = 1.0
    w_recon: float = 1.0
    w_res: float = 0.1
    val_every: int = 2000
    log_every: int = 200
    checkpoint_every: int = 3000
cfg = Cfg()
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed); _rnd.seed(cfg.seed)
print('config OK')


## 5. Dataset (CAVE)

In [ ]:
def load_mat_any(path):
    """Read a .mat array, falling back to h5py (v7.3/HDF5) files."""
    try:
        import scipy.io as sio
        mat = sio.loadmat(path)
        for k, v in mat.items():
            if not k.startswith('__') and isinstance(v, np.ndarray) and v.ndim >= 2:
                return np.asarray(v)
    except NotImplementedError:
        pass
    import h5py
    with h5py.File(path, 'r') as f:
        for k in f.keys():
            v = f[k]
            if isinstance(v, h5py.Dataset):
                arr = np.asarray(v)
                # h5py gives (B, H, W) as-is; sometimes transposed
                return arr
    raise ValueError(f'no array in {path}')

def discover_layouts():
    """Return dict name -> root for every attached dataset.
    Walks /kaggle/input recursively (handles nested layouts like
    /kaggle/input/datasets/liptee/...) and classifies by name hint."""
    import glob as _g
    from hsifusion.io_utils import find_dataset_roots, SPLIT_NAMES, TEST_NAMES
    out = {}
    input_root = '/kaggle/input'
    if not os.path.isdir(input_root):
        return out
    # 1) structured datasets found by the recursive walker
    candidates = []
    for base in (input_root,):
        if os.path.isdir(base):
            candidates += find_dataset_roots(base, max_depth=6)
    print('mounted dataset roots:', [os.path.relpath(r, input_root) for r in candidates])
    def find_sub(base, parts):
        node = base
        for p in parts:
            hit = None
            if os.path.isdir(os.path.join(node, p)):
                hit = os.path.join(node, p)
            else:
                for cand in os.listdir(node):
                    if os.path.isdir(os.path.join(node, cand)) and cand.lower() == p.lower():
                        hit = os.path.join(node, cand)
                        break
            if hit is None:
                return None
            node = hit
        return node
    for root in candidates:
        full_low = root.lower()
        key = None
        if 'cave' in full_low:
            key = 'CAVE'
        elif 'harvard' in full_low:
            key = 'HARVARD'
        if key is None or key in out:
            continue
        if key == 'CAVE':
            hsi = find_sub(root, ['Train', 'HSI'])
            if hsi is None:
                hsi = find_sub(root, ['Test', 'HSI'])
            if hsi is not None:
                out['CAVE'] = os.path.dirname(os.path.dirname(hsi))
        elif key == 'HARVARD':
            hsi = find_sub(root, ['Data', 'Test', 'HSI'])
            if hsi is None:
                hsi = find_sub(root, ['Test', 'HSI'])
            if hsi is not None:
                out['HARVARD'] = os.path.dirname(os.path.dirname(hsi))
    # 2) single-scene .mat datasets (Chikusei / Pavia) found by direct glob
    for ref, hint in [('CHIKUSEI', 'chikusei'), ('PAVIA', 'pavia')]:
        if ref in out:
            continue
        mats = sorted(_g.glob(os.path.join(input_root, '**', '*.mat'), recursive=True))
        for m in mats:
            if hint in m.lower():
                out[ref] = os.path.dirname(m)
                break
    return out

class DatasetSpec:
    """A dataset with train/test scene lists, band count and single-mat support."""
    def __init__(self, name, root, bands):
        self.name = name
        self.root = root
        self.bands = bands
        self.single_mat = None
        self._train, self._test = None, None

    def as_single(self, mat_path, patch=256, frac=0.7, seed=42):
        """Split one big scene into non-overlapping patches; first `frac` train."""
        arr = load_mat_any(mat_path)
        if arr.ndim == 2:
            arr = arr[None]
        a = np.squeeze(arr).astype(np.float32)
        if a.shape[0] == self.bands or a.shape[0] < a.shape[-1]:
            a = a  # already CHW
        elif a.shape[-1] == self.bands:
            a = np.transpose(a, (2, 0, 1))
        mx = float(a.max())
        if mx > 1.0:
            a = a / mx
        a = np.clip(a, 0.0, 1.0)
        H, W = a.shape[1], a.shape[2]
        patch = min(patch, H, W)
        ph = H // patch * patch
        pw = W // patch * patch
        patches = []
        for y in range(0, ph, patch):
            for x in range(0, pw, patch):
                patches.append(a[:, y:y + patch, x:x + patch])
        if not patches:
            patches = [a]
        rng = np.random.RandomState(seed)
        idx = rng.permutation(len(patches))
        n_tr = max(1, int(frac * len(patches)))
        n_tr = min(n_tr, len(patches) - 1) if len(patches) > 1 else 1
        tr_idx, te_idx = idx[:n_tr], idx[n_tr:]
        if len(te_idx) == 0:
            te_idx = tr_idx[:1]
        if len(tr_idx) == 0:
            tr_idx = te_idx[:1]
        self.single_mat = None  # do not retain the full-resolution cube
        self._train = [('patch%02d' % i, np.array(patches[j], copy=True)) for i, j in enumerate(tr_idx)]
        self._test = [('patch%02d' % i, np.array(patches[j], copy=True)) for i, j in enumerate(te_idx)]
        return self

    @property
    def train(self):
        if self._train is None:
            self._train = list_hsi(self.root, 'Train')
        return self._train

    @property
    def test(self):
        if self._test is None:
            try:
                self._test = list_hsi(self.root, 'Test')
            except FileNotFoundError:
                self._test = list_hsi(self.root, 'Train')
        return self._test

def build_specs():
    from hsifusion.io_utils import load_mat
    from hsifusion.io_utils import load_mat as _lm
    roots = discover_layouts()
    specs = []
    if 'CAVE' in roots:
        from hsifusion.io_utils import infer_channels
        b, m = infer_channels(roots['CAVE'])
        specs.append(DatasetSpec('CAVE', roots['CAVE'], b))
    if 'HARVARD' in roots:
        # band count from the first scene (31 for Harvard)
        all_scenes = list_hsi(roots['HARVARD'], 'Test')
        b = int(min(np.squeeze(load_mat_any(all_scenes[0][1])).shape))
        spec = DatasetSpec('HARVARD', roots['HARVARD'], b)
        # no Train split in harvard-hsi-2 -> explicit deterministic split
        rng = np.random.RandomState(42)
        perm = rng.permutation(len(all_scenes))
        n_tr = max(1, int(0.6 * len(all_scenes)))
        spec._train = [all_scenes[j] for j in perm[:n_tr]]
        spec._test = [all_scenes[j] for j in perm[n_tr:]]
        specs.append(spec)
    for ref, hint, patch in [('CHIKUSEI', 'chikusei', 128), ('PAVIA', 'pavia', 64)]:
        if ref in roots:
            import glob as _g
            mats = sorted(_g.glob(os.path.join(roots[ref], '*.mat')))
            if mats:
                a = np.squeeze(load_mat_any(mats[0]))
                b = int(min(a.shape))
                if b not in (100, 101, 102, 103, 104, 105, 126, 127, 128, 129, 130):
                    b = int(min(a.shape))
                spec = DatasetSpec(ref, roots[ref], b).as_single(mats[0], patch=patch)
                specs.append(spec)
    return specs, roots

specs, roots = build_specs()
for s in specs:
    print(f'{s.name:10s} bands={s.bands:<4} train={len(s.train):>4} test={len(s.test):>4} '
          f'root={os.path.basename(s.root)}')


In [ ]:
def get_cave_spec():
    layouts = discover_layouts()
    print('layouts:', list(layouts.keys()))
    if 'CAVE' in layouts:
        return DatasetSpec('CAVE', layouts['CAVE'], bands=31), 'CAVE'
    for name, root in layouts.items():
        spec = DatasetSpec(name, root, bands=31)
        try:
            if len(spec.train) > 0:
                return spec, name
        except Exception:
            continue
    raise RuntimeError('no dataset found')
spec, ds_name = get_cave_spec()
print(f'{ds_name}: train={len(spec.train)} test={len(spec.test)} bands={spec.bands}')


In [ ]:
srf = nikon_d700_srf(spec.bands)
srf_t = torch.from_numpy(srf).float().to(DEVICE)
print('SRF shape:', srf.shape)


In [ ]:
def simulate_obs(gt_hsi, cfg, srf, sigma=1.2, noise=0.0):
    from hsifusion.degrade import FixedDegradation
    g = torch.from_numpy(np.ascontiguousarray(gt_hsi))[None].to(DEVICE)
    deg = FixedDegradation(cfg.scale, cfg.blur_ksize, sigma).to(DEVICE)
    lr = deg(g)
    srf_t = torch.from_numpy(srf).to(DEVICE)
    msi = torch.einsum('chw,cm->mhw', g[0], srf_t)
    if noise > 0:
        msi = msi + noise * msi.std() * torch.randn_like(msi)
        lr = lr + noise * lr.std() * torch.randn_like(lr)
    return lr, msi, g
print('simulate_obs OK')


## 6. Training (time-budgeted, EMA, physics loss)

In [ ]:
band_cache = {}
def get_hsi(stem, hp):
    if stem not in band_cache:
        band_cache[stem] = hp if isinstance(hp, np.ndarray) else load_hsi(hp, spec.bands)
    return band_cache[stem]

model = KrylovNet(bands=spec.bands, msi_bands=3, scale=cfg.scale, rho=1e-3,
                  n_stages=6, prior_width=96, prior_blocks=8, n_outer=4,
                  blur_ksize=cfg.blur_ksize, eval_sigma=cfg.eval_sigma)
model.to(DEVICE)
model.set_srf(srf_t)
nparams = sum(p.numel() for p in model.parameters())
print(f'KrylovNet params: {nparams}')


In [ ]:
def random_kernel(batch, ksize=9, s_range=(0.6, 2.4), aniso=0.5):
    ks = torch.empty(batch).uniform_(*s_range)
    out = []
    for i in range(batch):
        sy = ks[i]
        sx = ks[i] * torch.empty(1).uniform_(0.8, 1.25) if torch.rand(1) < aniso else ks[i]
        th = torch.empty(1).uniform_(0, 3.1416)
        out.append(gaussian_kernel2d(ksize, sx, sy, th))
    return torch.stack(out).to(DEVICE)

def sample_batch(patch=96, bs=12):
    lrs, msis, gts, ks = [], [], [], []
    for _ in range(bs):
        stem, hp = spec.train[_rnd.randrange(len(spec.train))]
        hsi = get_hsi(stem, hp)
        H, W = hsi.shape[1], hsi.shape[2]
        p = min(patch, H, W)
        Hp = (H // p) * p
        y = _rnd.randrange(0, H - Hp + 1)
        x = _rnd.randrange(0, W - Hp + 1)
        gt = np.ascontiguousarray(hsi[:, y:y + p, x:x + p])
        if _rnd.random() < 0.5:
            gt = gt[:, :, ::-1].copy()
        if _rnd.random() < 0.5:
            gt = gt[:, ::-1, :].copy()
        sig = _rnd.uniform(*cfg.sigma_range)
        k = gaussian_kernel2d(cfg.blur_ksize, sig, sig, 0.0)
        lr, msi, _ = simulate_obs(gt, cfg, srf, sigma=sig,
                                  noise=_rnd.uniform(0, 0.02))
        lrs.append(lr[0].cpu()); msis.append(msi[0].cpu())
        gts.append(torch.from_numpy(gt)); ks.append(k.cpu())
    return (torch.stack(lrs).to(DEVICE), torch.stack(msis).to(DEVICE),
            torch.stack(gts).to(DEVICE), torch.stack(ks).to(DEVICE))


In [ ]:
class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point:
                    self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
    def apply_to(self, model):
        model.load_state_dict(self.shadow, strict=False)
    def restore_from(self, model):
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

def krylov_loss(pred, gt, lr, msi, kernel, model):
    out = pred['out']
    op = model.op
    l_phys = F.mse_loss(op.D(out, kernel), lr)
    l_spec = F.mse_loss(op.S(out, model.srf), msi)
    l_recon = F.l1_loss(out, gt)
    l_res = pred['residuals'][-1].mean()
    total = (cfg.w_phys * l_phys + cfg.w_spec * l_spec
             + cfg.w_recon * l_recon + cfg.w_res * l_res)
    return total, l_phys, l_spec, l_recon, l_res


In [ ]:
@torch.no_grad()
def validate(scenes, verbose=False):
    model.eval()
    agg = {'psnr': [], 'ssim': [], 'sam': [], 'ergas': []}
    rows = []
    for stem, hp in scenes:
        hsi = hp if isinstance(hp, np.ndarray) else load_hsi(hp, spec.bands)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        lr, msi, gt = simulate_obs(hsi[:, :h, :w], cfg, srf)
        pred = model(lr, msi)
        m = evaluate_arrays(pred['out'][0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append({'scene': stem, **m})
        for k_, v in m.items():
            agg[k_].append(v)
        if verbose:
            print(f'  {stem:<24} PSNR={m["psnr"]:7.3f}  SSIM={m["ssim"]:.4f}  '
                  f'SAM={m["sam"]:6.3f}  ERGAS={m["ergas"]:8.3f}')
    return {k_: float(np.mean(v)) for k_, v in agg.items()}, rows


In [ ]:
def lr_at(it, total, warm, lr0, min_lr):
    if it < warm:
        return lr0 * it / warm
    return max(min_lr, 0.5 * lr0 * (1 + np.cos(np.pi * (it - warm) / (total - warm))))

opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
scaler = GradScaler(enabled=cfg.amp)
ema = EMA(model, cfg.ema_decay)
total, warm = cfg.iters, cfg.warmup

resume = None
import glob as _glob
cands = _glob.glob('/kaggle/input/krylovnet-cp/checkpoint.pt')
if cands:
    try:
        resume = torch.load(cands[0], map_location=DEVICE)
        print('resume file found, iter', resume.get('iter'))
    except Exception as e:
        print('resume load failed:', e)
        resume = None
it0, best_psnr, best_state, history = 0, -1, None, []
if resume:
    model.load_state_dict(resume['model'])
    ema.shadow = {k: v.to(DEVICE) for k, v in resume['ema'].items()}
    opt.load_state_dict(resume['opt'])
    it0 = int(resume['iter']) + 1
    best_psnr = float(resume['best_psnr'])
    best_state = resume['best_state']
    history = resume['history']
    print(f'RESUMED from iter {resume["iter"]} (best {best_psnr:.3f} dB)')
else:
    print('fresh start')

CP_WORK = '/kaggle/working/cp'
os.makedirs(CP_WORK, exist_ok=True)
CP_FILE = os.path.join(CP_WORK, 'checkpoint.pt')
with open(os.path.join(CP_WORK, 'dataset-metadata.json'), 'w') as f:
    json.dump({'id': 'amarnathmadaka/krylovnet-cp',
               'title': 'KrylovNet CAVE checkpoint',
               'licenses': [{'name': 'other'}]}, f)

def save_checkpoint(it):
    torch.save({'model': {k: v.detach().clone() for k, v in model.state_dict().items()},
                'ema': ema.shadow, 'opt': opt.state_dict(), 'iter': it,
                'best_psnr': best_psnr, 'best_state': best_state,
                'history': history, 'cfg': asdict(cfg), 'nparams': nparams},
               CP_FILE)
    note = f'iter {it} best {best_psnr:.3f}'
    try:
        out = _sp.run(['kaggle', 'datasets', 'version', '-p', CP_WORK, '-m', note],
                      capture_output=True, text=True, timeout=600)
        if out.returncode != 0:
            out = _sp.run(['kaggle', 'datasets', 'create', '-p', CP_WORK, '-m', note],
                          capture_output=True, text=True, timeout=600)
        tail = (out.stdout or out.stderr or '').strip().splitlines()[-1][:150]
        print(f'[ckpt] {tail}')
    except Exception as e:
        print('[ckpt] upload failed:', e)

T0 = time.time()
TIME_LIMIT = cfg.time_budget_h * 3600.0
print('training start')


In [ ]:
try:
    for it in range(it0, total + 1):
        if time.time() - T0 > TIME_LIMIT:
            print(f'[time budget reached at iter {it}]')
            break
        opt.param_groups[0]['lr'] = lr_at(it, total, warm, cfg.lr, cfg.min_lr)
        model.train()
        lr, msi, gt, k = sample_batch()
        opt.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.amp):
            pred = model(lr, msi, k)
            loss, lp, ls, lr_, lres = krylov_loss(pred, gt, lr, msi, k, model)
        total_loss = loss / cfg.grad_accum
        scaler.scale(total_loss).backward()
        if (it + 1) % cfg.grad_accum == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            ema.update(model)
        if it % cfg.log_every == 0:
            print(f'[{it}/{total}] loss={loss.item():.4f} phys={lp.item():.3f} '
                  f'spec={ls.item():.3f} recon={lr_.item():.4f} res={lres.item():.4f} '
                  f'({time.time() - T0:.0f}s)')
        if (it + 1) % cfg.val_every == 0 or it == total:
            ema.apply_to(model)
            vm, _ = validate(spec.test[:4])
            print(f'[val @ {it}] PSNR={vm["psnr"]:.3f} SSIM={vm["ssim"]:.4f} '
                  f'SAM={vm["sam"]:.3f} ERGAS={vm["ergas"]:.3f}')
            history.append({'iter': it, **vm})
            if vm['psnr'] > best_psnr:
                best_psnr = vm['psnr']
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                print(f'[save] best PSNR={best_psnr:.3f}')
            ema.restore_from(model)
        if (it + 1) % cfg.checkpoint_every == 0:
            ema.apply_to(model)
            save_checkpoint(it)
            ema.restore_from(model)
except KeyboardInterrupt:
    print('interrupted')
save_checkpoint(it)
print('training done')


## 7. Final evaluation (best model, full test split)

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)
model.eval()
mean, rows = validate(spec.test, verbose=True)
print('=' * 60)
print(f'FINAL {ds_name} TEST: PSNR={mean["psnr"]:.3f} SSIM={mean["ssim"]:.4f} '
      f'SAM={mean["sam"]:.3f} ERGAS={mean["ergas"]:.3f}')
print('=' * 60)
result = {'dataset': ds_name, 'protocol': 'Nikon D700 SRF, Wald blur, x4',
          'nparams': nparams, 'mean': mean, 'rows': rows,
          'history': history, 'cfg': asdict(cfg), 'iters_run': it}
with open('sota_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print('saved sota_results.json')
